## High level analysis

- Number of job adverts
- Number of job adverts per occupation
- Number of job adverts per region
- Salaries
- Outliers

In [1]:
import pandas as pd
from dap_prinz_green_jobs.getters.data_getters import load_s3_data
from dap_prinz_green_jobs import BUCKET_NAME, analysis_config
from dap_prinz_green_jobs.utils.chloropleth_utils import get_nuts2polygons_dict, get_nuts1polygons_dict, get_nuts3polygons_dict
from dap_prinz_green_jobs.getters.industry_getters import load_sic
import dap_prinz_green_jobs.analysis.ojo_analysis.process_ojo_green_measures as pg

import altair as alt
import ast

In [2]:
occ_agg_gje = pd.read_csv(
    f"s3://prinz-green-jobs/outputs/data/ojo_application/extracted_green_measures/analysis/{analysis_config['analysis_files']['agg_soc_date_stamp']}/occupation_aggregated_data_{analysis_config['analysis_files']['agg_soc_date_stamp']}_extra_gjeformat.csv")

len(occ_agg_gje)

2024-11-25 11:15:42,722 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


1095

In [3]:
occ_agg_gje['num_job_ads'].min()

52

In [5]:
occ_agg = pd.read_csv(
    f"s3://prinz-green-jobs/outputs/data/ojo_application/extracted_green_measures/analysis/{analysis_config['analysis_files']['agg_soc_date_stamp']}/occupation_aggregated_data_{analysis_config['analysis_files']['agg_soc_date_stamp']}_all.csv")


In [6]:
occ_agg[occ_agg['clean_soc_name']=='Betting shop managers']

,SOC_2020_EXT,num_job_ads,prop_job_ads,top_5_socs,occ_timeshare,occ_topics,average_occ_green_timeshare,average_num_skills,average_prop_green_skills,top_5_green_skills,...,SOC_2020_EXT_name,clean_soc_name,soc_description,SOC_2020,SOC_2010,green_topics_lists,occ_greenness,ind_greenness,skills_greenness,greenness_score
125,1256/01,179,0.00003,"[{'soc_id': '1256/01', 'soc_name': 'Betting sh...",39.6,4,39.6,10.826816,0.098668,[{'skill_name': 'implement environmental prote...,...,Betting shop managers,Betting shop managers,Betting shop managers are responsible for all ...,1256,1259,"['Green recreation', 'Regulation enforcement',...",high,high,high,high


In [7]:
occ_agg = pd.read_csv(
    f"s3://prinz-green-jobs/outputs/data/ojo_application/extracted_green_measures/analysis/{analysis_config['analysis_files']['agg_soc_date_stamp']}/occupation_aggregated_data_{analysis_config['analysis_files']['agg_soc_date_stamp']}_all.csv")

occ_agg = occ_agg[occ_agg['clean_soc_name']!='Betting shop managers']
occ_agg.reset_index(inplace=True)

In [13]:
itl_aggregated_data = load_s3_data(
        BUCKET_NAME,
        f"outputs/data/ojo_application/extracted_green_measures/analysis/{analysis_config['analysis_files']['agg_region_date_stamp']}/all_itl_aggregated_data_{analysis_config['analysis_files']['agg_region_date_stamp']}.csv"
        )

itl_aggregated_data['average_perc_green_skills'] = itl_aggregated_data['average_prop_green_skills']*100
itl_aggregated_data['average_prop_occ_green_timeshare'] = itl_aggregated_data['average_occ_green_timeshare']/100

itl_aggregated_data.rename(columns = {'itl_level': "itl_type"}, inplace=True)

2024-11-25 11:18:01,806 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [14]:
nuts1polygons_dict = get_nuts1polygons_dict()
itl1polygons_dict = {k.replace("UK","TL"):v for k, v in nuts1polygons_dict.items()}

nuts2polygons_dict = get_nuts2polygons_dict()
itl2polygons_dict = {k.replace("UK","TL"):v for k, v in nuts2polygons_dict.items()}

nuts3polygons_dict = get_nuts3polygons_dict()
itl3polygons_dict = {k.replace("UK","TL"):v for k, v in nuts3polygons_dict.items()}

allpolygons_dict = {**itl1polygons_dict, **itl2polygons_dict, **itl3polygons_dict}

In [15]:
# Just using this to get the ITL names
itl_aggregated_data['geometry_name'] = itl_aggregated_data["itl_code"].map(allpolygons_dict)
itl_aggregated_data[['geometry', 'itl_name']] = itl_aggregated_data['geometry_name'].apply(lambda x: pd.Series(x))
itl_aggregated_data.drop('geometry_name', axis=1, inplace=True)
itl_aggregated_data.drop('geometry', axis=1, inplace=True)

In [16]:
itl1_aggregated_data = itl_aggregated_data[itl_aggregated_data['itl_type']=='itl_1_code']
itl2_aggregated_data = itl_aggregated_data[itl_aggregated_data['itl_type']=='itl_2_code']
itl3_aggregated_data = itl_aggregated_data[itl_aggregated_data['itl_type']=='itl_3_code']

## Occupations

In [17]:
print(f"There are {occ_agg['num_job_ads'].sum()} job adverts assigned to {len(occ_agg)} SOC 6-digit occupations")
print(f"These are from {occ_agg['SOC_2020'].nunique()} unique SOC 4-digit codes")
print(f"There are {sum(occ_agg['num_job_ads']>50)} ({round(sum(occ_agg['num_job_ads']>50)*100/len(occ_agg),2)})% SOC 6-digit occupations with over 50 job adverts")
print(f"There are {sum(occ_agg['num_job_ads']>100)} ({round(sum(occ_agg['num_job_ads']>100)*100/len(occ_agg),2)})% SOC 6-digit occupations with over 100 job adverts")

There are 5049269 job adverts assigned to 1337 SOC 6-digit occupations
These are from 412 unique SOC 4-digit codes
There are 1095 (81.9)% SOC 6-digit occupations with over 50 job adverts
There are 986 (73.75)% SOC 6-digit occupations with over 100 job adverts


In [18]:
print("In the GJE..")
print(f"There are {occ_agg_gje['num_job_ads'].sum()} job adverts assigned to {len(occ_agg_gje)} SOC 6-digit occupations")
print(f"These are from {occ_agg_gje['SOC_2020'].nunique()} unique SOC 4-digit codes")

In the GJE..
There are 5044917 job adverts assigned to 1095 SOC 6-digit occupations
These are from 409 unique SOC 4-digit codes


In [19]:
thresh = 50
a = alt.Chart(
    occ_agg[occ_agg['num_job_ads']<=thresh], title=f"Number of job adverts <= {thresh}"
         ).mark_bar(color='blue').encode(
    alt.X("num_job_ads", title= "Number of job adverts (binned)", bin=True),
    y=alt.Y('count()', title="Number of occupations"),
)

b = alt.Chart(
    occ_agg[occ_agg['num_job_ads']>thresh], title=f"Number of job adverts > {thresh}",
         ).mark_bar(color='blue').encode(
    alt.X("num_job_ads", title= "Number of job adverts (binned)", bin=alt.BinParams(maxbins=40)),
    y=alt.Y('count()', title="Number of occupations").scale(type="log"),
)

a |b

alt.HConcatChart(...)

In [20]:
occ_agg[occ_agg['num_job_ads']>50]['num_job_ads'].quantile(.25)

248.5

In [21]:
occ_agg[occ_agg['num_job_ads']>50]['num_job_ads'].median()

919.0

In [22]:
occ_agg[occ_agg['num_job_ads']>50]['num_job_ads'].quantile(.5)

919.0

In [23]:
occ_agg[occ_agg['num_job_ads']>50]['num_job_ads'].quantile(.75)

3654.5

In [24]:
sum(occ_agg['num_job_ads'].between(50,200))/sum(occ_agg['num_job_ads']>50)

0.2182648401826484

In [25]:
sum(occ_agg['num_job_ads'].between(50,5000))/sum(occ_agg['num_job_ads']>50)

0.7990867579908676

In [26]:
print(occ_agg[occ_agg['num_job_ads']==1]['clean_soc_name'].tolist())

['Alexander technique teachers', 'Homeopaths (excludes medically qualified)', 'Hypnotherapists', 'Antenatal teachers', 'Hand drawn animators', 'Tattoo and henna artists', 'Authors', 'Dancers', 'Performance make-up artists', 'Estate agents and auctioneers', 'Beekeepers', 'Falconers', 'Fish and river keepers', 'Fishers', 'Gunsmiths', 'Monumental masons', 'Thatchers', 'Dressmakers', 'Milliners (excludes wholesale, retail trade)', 'Film and television runners', 'Weight loss advisers', 'Forestry and related workers', 'Drain cleaners', 'Clairvoyants, mediums and astrologers']


In [27]:
occ_agg[occ_agg['num_job_ads']<=50].sort_values(by='num_job_ads')[['clean_soc_name', 'num_job_ads']][0:10]

,clean_soc_name,num_job_ads
1335,"Clairvoyants, mediums and astrologers",1
991,"Milliners (excludes wholesale, retail trade)",1
989,Dressmakers,1
942,Monumental masons,1
541,Alexander technique teachers,1
542,Homeopaths (excludes medically qualified),1
544,Hypnotherapists,1
548,Antenatal teachers,1
895,Gunsmiths,1
1094,Film and television runners,1


In [28]:
num_thresh = []
for thresh in range(0,1000):
    num_thresh.append({
        "Minimum number of job adverts": thresh,
        "Number of occupations":sum(occ_agg['num_job_ads']>thresh),
        "Proportion of occupations":sum(occ_agg['num_job_ads']>thresh)/len(occ_agg)
    })

In [29]:
alt.Chart(
    pd.DataFrame(num_thresh)
         ).mark_line(color='blue').encode(
    alt.X("Minimum number of job adverts"),
    y=alt.Y('Proportion of occupations'),
)

alt.Chart(...)

# Salaries
- In data for GjE

In [30]:
occ_agg_gje['median_min_annualised_salary'].max()

117000.0

In [31]:
occ_agg_gje[occ_agg_gje['median_min_annualised_salary']>100000][['clean_soc_name', 'median_min_annualised_salary']]

,clean_soc_name,median_min_annualised_salary
816,Educational psychologists,117000.0
1055,General practitioners,110370.0


In [32]:
occ_agg_sal_filt = occ_agg_gje[occ_agg_gje['median_min_annualised_salary']<=110000]
occ_agg_sal_filt['binned_median_min_annualised_salary'] = pd.cut(occ_agg_sal_filt['median_min_annualised_salary'], bins=10)

dd = occ_agg_sal_filt.groupby(['binned_median_min_annualised_salary'])['num_job_ads'].sum().reset_index()
dd['binned_median_min_annualised_salary_str'] = dd['binned_median_min_annualised_salary'].astype(str)
alt.Chart(dd[['num_job_ads', 'binned_median_min_annualised_salary_str']]).mark_bar(color='blue').encode(
    x=alt.X('num_job_ads', title="Number of job adverts"), # .scale(type="symlog")
    y=alt.Y('binned_median_min_annualised_salary_str', title="Minimum salary (binned)",sort=None),
    tooltip=[
        alt.Tooltip("num_job_ads", title="Number of job adverts"), 
    ]  
)

/var/folders/xc/s255_bsx0l7cbx43t290kjtr0000gn/T/ipykernel_66651/1669039397.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  occ_agg_sal_filt['binned_median_min_annualised_salary'] = pd.cut(occ_agg_sal_filt['median_min_annualised_salary'], bins=10)


alt.Chart(...)

In [33]:
ddj = occ_agg_sal_filt.groupby(['binned_median_min_annualised_salary'])['SOC_2020_EXT_name'].nunique().reset_index()
ddj['binned_median_min_annualised_salary_str'] = ddj['binned_median_min_annualised_salary'].astype(str)
alt.Chart(ddj[['SOC_2020_EXT_name', 'binned_median_min_annualised_salary_str']]).mark_bar(color='blue').encode(
    x=alt.X('SOC_2020_EXT_name', title="Number of occupations"), # .scale(type="symlog")
    y=alt.Y('binned_median_min_annualised_salary_str', title="Minimum salary (binned)",sort=None),
    tooltip=[
        alt.Tooltip("SOC_2020_EXT_name", title="Number of occupations"), 
    ]  
)

alt.Chart(...)

## Number of skills
- Do some occupations have very small numbers of skills?

In [34]:
occ_agg_gje['average_prop_green_skills'].median()

0.0046522407633518

In [35]:
occ_agg_gje['average_num_skills'].median()

13.070886075949367

In [36]:
occ_agg_gje['average_num_skills'].quantile(0.25)

9.474279430467861

In [37]:
occ_agg_gje['average_num_skills'].quantile(0.75)

15.987074829931972

In [38]:
thresh = 5
print(f"There are {len(occ_agg_gje[occ_agg_gje['average_num_skills']<thresh])} occupations with less than an average of {thresh} skills per job advert")

print("There occupations are..")
print(occ_agg_gje[occ_agg_gje['average_num_skills']<thresh]['SOC_2020_EXT_name'].unique().tolist())

There are 12 occupations with less than an average of 5 skills per job advert
There occupations are..
['Bricklayers', "Builder's labourers", 'Scaffolders and stagers', 'Groundworkers   ', 'Directors in logistics, warehousing and transport n.e.c.', 'Dry liners', 'Steel fixers and underpinners', 'Anaesthetists', 'Smart energy experts', 'Bicycle and motorcycle couriers', 'Mystery shoppers', 'Ceiling fitters']


In [39]:
occ_agg_gje[occ_agg_gje['average_num_skills']<thresh][['SOC_2020_EXT_name', 'average_prop_green_skills']]

,SOC_2020_EXT_name,average_prop_green_skills
256,Bricklayers,0.011244
294,Builder's labourers,0.009819
381,Scaffolders and stagers,0.007594
478,Groundworkers,0.005663
621,"Directors in logistics, warehousing and transp...",0.003710
764,Dry liners,0.002545
771,Steel fixers and underpinners,0.002490
986,Anaesthetists,0.001195
1034,Smart energy experts,0.000809
1060,Bicycle and motorcycle couriers,0.000421


In [40]:

occ_agg_gje['binned_average_num_skills'] = pd.cut(occ_agg_gje['average_num_skills'], bins=10)

dd = occ_agg_gje.groupby(['binned_average_num_skills'])['SOC_2020_EXT_name'].nunique().reset_index()
dd['binned_average_num_skills_str'] = dd['binned_average_num_skills'].astype(str)
alt.Chart(dd[['SOC_2020_EXT_name', 'binned_average_num_skills_str']]).mark_bar(color='blue').encode(
    x=alt.X('SOC_2020_EXT_name', title="Number of occupations"), # .scale(type="symlog")
    y=alt.Y('binned_average_num_skills_str', title="Average number of skills",sort=None),
    tooltip=[
        alt.Tooltip("SOC_2020_EXT_name", title="Number of occupations"), 
    ]  
)

alt.Chart(...)

## Regions

- ITL 1 and 2 all have over 100 job adverts

In [41]:
print(f"There are {itl1_aggregated_data['num_job_ads'].sum()} job adverts assigned to {itl1_aggregated_data['itl_code'].nunique()} ITL1 regions")
print(f"There are {itl2_aggregated_data['num_job_ads'].sum()} job adverts assigned to {itl2_aggregated_data['itl_code'].nunique()} ITL2 regions")
print(f"There are {itl3_aggregated_data['num_job_ads'].sum()} job adverts assigned to {itl3_aggregated_data['itl_code'].nunique()} ITL3 regions")

There are 5794975 job adverts assigned to 12 ITL1 regions
There are 5752595 job adverts assigned to 37 ITL2 regions
There are 5752595 job adverts assigned to 159 ITL3 regions


In [42]:
print(f"{sum(itl1_aggregated_data['num_job_ads']>1000)/len(itl1_aggregated_data)} ITL 1 regions have over 1000 job adverts")
print(f"{sum(itl2_aggregated_data['num_job_ads']>1000)/len(itl2_aggregated_data)} ITL 2 regions have over 1000 job adverts")
print(f"{sum(itl3_aggregated_data['num_job_ads']>1000)/len(itl3_aggregated_data)} ITL 3 regions have over 1000 job adverts")
print(f"{sum(itl3_aggregated_data['num_job_ads']>100)/len(itl3_aggregated_data)} ITL 3 regions have over 100 job adverts")

1.0 ITL 1 regions have over 1000 job adverts
1.0 ITL 2 regions have over 1000 job adverts
0.9245283018867925 ITL 3 regions have over 1000 job adverts
1.0 ITL 3 regions have over 100 job adverts


In [43]:
itl3_aggregated_data.sort_values(by='num_job_ads')[0:10][['itl_name', 'num_job_ads']]

,itl_name,num_job_ads
51,Darlington,147
178,Na h-Eileanan Siar (Western Isles),440
180,Shetland Islands,493
203,Causeway Coast and Glens,530
179,Orkney Islands,601
200,Ards and North Down,661
201,Derry City and Strabane,686
206,Mid and East Antrim,741
204,Antrim and Newtownabbey,798
205,Lisburn and Castlereagh,823


In [44]:
num_thresh_itl3 = []
for thresh in range(0,1000):
    num_thresh_itl3.append({
        "Minimum number of job adverts": thresh,
        "Number of regions":sum(itl3_aggregated_data['num_job_ads']>thresh),
        "Proportion of regions":sum(itl3_aggregated_data['num_job_ads']>thresh)/len(itl3_aggregated_data)
    })

alt.Chart(
    pd.DataFrame(num_thresh_itl3)
         ).mark_line(color='blue').encode(
    alt.X("Minimum number of job adverts"),
    y=alt.Y('Proportion of regions'),
)

alt.Chart(...)

In [45]:
alt.Chart(
    itl1_aggregated_data.sort_values(by="num_job_ads", ascending=False)
         ).mark_bar(color='blue').encode(
    x=alt.X('num_job_ads', title="Number of job adverts"),
    y = alt.Y("itl_name", title="ITL 1 region name", sort=None),
    tooltip=[
        alt.Tooltip("itl_name", title="Region"),
        alt.Tooltip("num_job_ads", title="Number of job adverts"),
        alt.Tooltip("prop_job_ads", title="Percentage of job adverts", format=".2%"),   
    ]  
)

alt.Chart(...)

In [46]:
alt.Chart(
    itl2_aggregated_data.sort_values(by="num_job_ads", ascending=False)
         ).mark_bar(color='blue').encode(
    x=alt.X('num_job_ads', title="Number of job adverts"),
    y = alt.Y("itl_name", title="ITL 2 region name", sort=None),
    tooltip=[
        alt.Tooltip("itl_name", title="Region"),
        alt.Tooltip("num_job_ads", title="Number of job adverts"),
        alt.Tooltip("prop_job_ads", title="Percentage of job adverts", format=".2%"),   
    ]  
)

alt.Chart(...)